# 03 · Joins: Vendas + Funcionários + Empresas (Caso A)

**Teoria**: docs/04-dataframes-catalyst-tungsten.md

As duas perguntas que só um join resolve: **quem mais vendeu?** e
**qual setor vendeu mais em cada período?**. A segunda encadeia
`vendas → funcionarios → empresas`, porque `setor` só existe em
`empresas`.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

spark = get_local_session("03-joins")
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))

## Ranking de funcionários por vendas

`vendas` só tem `id_funcionario` — para ver o nome, precisamos do join
com `funcionarios`.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as spark_sum

ranking_funcionarios = (
    vendas.join(funcionarios, "id_funcionario")
    .groupBy("nome_funcionario", "cargo")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
ranking_funcionarios.show(15, truncate=False)

## Total de vendas por setor e período (join encadeado)

`setor` vive em `empresas`, que só se conecta a `vendas` através de
`funcionarios`. Este é o join "de verdade", em duas etapas.

In [ ]:
vendas_por_setor_encadeado = (
    vendas.join(funcionarios, "id_funcionario")
    .join(empresas, "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_encadeado.show(15)

## O mesmo resultado, pelo atalho denormalizado

`vendas` já carrega `id_empresa` (o empregador do funcionário daquela
venda, gravado no momento da geração dos dados) — então dá pra pular direto
para `empresas`, sem passar por `funcionarios`. Duas rotas, mesmo
destino; compare as somas de `total_vendas` entre as duas células.

In [ ]:
from pyspark.sql.functions import broadcast

vendas_por_setor_direto = (
    vendas.join(broadcast(empresas), "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_direto.show(15)

## Prévia: por que o segundo join foi `broadcast`?

`empresas` tem só 50 linhas — cabe inteira na memória de cada executor,
então o Spark evita embaralhar ("shuffle") a tabela grande de `vendas`
pra fazer esse join. O notebook 04 mostra o plano de execução por trás
dessa escolha.

In [ ]:
spark.stop()